# XQuality / OLAF semantic validation metrics (STR, CR, PC, OC, CV, DVS)

This notebook computes the exact slide-style validation/alignment metrics:

| Metric | Direction | Meaning |
|---|---:|---|
| **STR** | ↑ | Support Traceability Rate: fraction of valid triples with explicit support/source-span metadata. |
| **CR** | ↓ | Conflict/Unresolved Ratio: fraction of raw rows that are invalid or unresolved (`?`, empty, missing source/target/relation). |
| **PC** | ↑ | Provenance Coverage: fraction of valid triples with provenance metadata. |
| **OC** | ↑ | Ontology Compliance: fraction of valid triples whose subject, predicate and object align to the ontology vocabulary. |
| **CV** | ↓ | Constraint Violation rate: fraction of valid triples that are not ontology-compliant or violate explicit domain/range constraints when available. |
| **DVS** | ↑ | Domain Validation Score: strict binary pass/fail based on thresholds over STR, CR, PC, OC and CV. |

Important: this is **not extraction precision/recall/F1**. It is a gold-free semantic validation/alignment evaluation for KG/ontology outputs.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)


def find_project_root(start=None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / 'src' / 'neoolaf').exists():
            return p
    for p in candidates:
        if (p / 'pyproject.toml').exists() and (p / 'examples' / 'XQualityMachine32').exists():
            return p
    raise RuntimeError('Could not find NeoOLAF project root. Run this notebook from inside the NeoOLAF repo.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

EXAMPLE_ROOT = PROJECT_ROOT / 'examples' / 'XQualityMachine32'
RUNS_ROOT = EXAMPLE_ROOT / 'runs'
MACHINE32_DIR = PROJECT_ROOT / 'data' / 'XQuality' / 'Machine32'
OUTPUT_DIR = RUNS_ROOT / 'olaf_semantic_validation_metrics'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('MACHINE32_DIR:', MACHINE32_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

if not MACHINE32_DIR.exists():
    raise FileNotFoundError(f'MACHINE32_DIR does not exist: {MACHINE32_DIR}')


## Import the metric implementation

Before running this cell, copy the patch files from `NeoOLAF_semantic_validation_metrics_patch.zip` into your NeoOLAF repo root.


In [ ]:
try:
    from neoolaf.evaluation.metrics.semantic_validation_metrics import (
        SemanticValidationConfig,
        load_triple_table,
        compute_semantic_validation_metrics,
    )
    print('Imported semantic validation metrics from src/neoolaf/evaluation/metrics/semantic_validation_metrics.py')
except Exception as exc:
    raise ImportError(
        'Could not import semantic_validation_metrics. Copy the patch into the repo root, '
        'restart the kernel, and run again. Original error: ' + repr(exc)
    )


## Locate OLAF files

The notebook searches under `data/XQuality/Machine32`. If your filenames differ, edit the variables in this cell.


In [ ]:
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

# Search recursively because your copied OLAF demo may be either directly in Machine32 or in output/data subfolders.
OLAF_TRIPLES_PATH = first_existing(
    list(MACHINE32_DIR.rglob('llm_alarm_triplets.xlsx'))
    + list(MACHINE32_DIR.rglob('*triplet*.xlsx'))
    + list(MACHINE32_DIR.rglob('*triple*.xlsx'))
    + list(MACHINE32_DIR.rglob('*kg*.json'))
)

OLAF_ONTOLOGY_PATH = first_existing(
    list(MACHINE32_DIR.rglob('llm_alarm_ontology.ttl'))
    + list(MACHINE32_DIR.rglob('*ontology*.ttl'))
    + list(MACHINE32_DIR.rglob('*ontology*.owl'))
    + list(MACHINE32_DIR.rglob('*.ttl'))
)

OLAF_SOURCE_TEXT_PATH = first_existing(
    list(MACHINE32_DIR.rglob('alarm.txt'))
    + list(MACHINE32_DIR.rglob('*.txt'))
)

print('OLAF_TRIPLES_PATH:', OLAF_TRIPLES_PATH)
print('OLAF_ONTOLOGY_PATH:', OLAF_ONTOLOGY_PATH)
print('OLAF_SOURCE_TEXT_PATH:', OLAF_SOURCE_TEXT_PATH)

if OLAF_TRIPLES_PATH is None:
    raise FileNotFoundError('Could not find OLAF triple file under data/XQuality/Machine32')
if OLAF_ONTOLOGY_PATH is None:
    raise FileNotFoundError('Could not find OLAF ontology file under data/XQuality/Machine32')


## Metric configuration

By default, STR requires **explicit support/provenance metadata**, not just lexical occurrence in the source text. This is stricter and closer to the slide interpretation: a system should carry traceability in the KG output itself.

If you want a weaker diagnostic STR based on source-text lexical matching, set `ALLOW_LEXICAL_SOURCE_SUPPORT_FOR_STR = True`.


In [ ]:
ALLOW_LEXICAL_SOURCE_SUPPORT_FOR_STR = False

CONFIG = SemanticValidationConfig(
    node_fuzzy_threshold=0.92,
    predicate_fuzzy_threshold=0.90,
    allow_lexical_source_support_for_str=ALLOW_LEXICAL_SOURCE_SUPPORT_FOR_STR,
    # Strict DVS thresholds. Adjust only if your slide definition differs.
    dvs_min_str=0.95,
    dvs_max_cr=0.05,
    dvs_min_pc=0.95,
    dvs_min_oc=0.95,
    dvs_max_cv=0.05,
)

print(CONFIG)


## Evaluate OLAF semantic validation metrics

In [ ]:
olaf_df = load_triple_table(OLAF_TRIPLES_PATH)
print('Raw OLAF table shape:', olaf_df.shape)
display(olaf_df.head(20))

olaf_summary, olaf_details_df = compute_semantic_validation_metrics(
    triples_df=olaf_df,
    ontology_path=OLAF_ONTOLOGY_PATH,
    source_text_path=OLAF_SOURCE_TEXT_PATH,
    config=CONFIG,
)

olaf_summary_df = pd.DataFrame([olaf_summary])
olaf_summary_df.insert(0, 'series', 'OLAF')
olaf_summary_df.insert(1, 'triple_path', str(OLAF_TRIPLES_PATH))
olaf_summary_df.insert(2, 'ontology_path', str(OLAF_ONTOLOGY_PATH))
olaf_summary_df.insert(3, 'source_text_path', str(OLAF_SOURCE_TEXT_PATH) if OLAF_SOURCE_TEXT_PATH else '')

summary_path = OUTPUT_DIR / 'olaf_semantic_validation_summary.csv'
details_path = OUTPUT_DIR / 'olaf_semantic_validation_details.csv'
olaf_summary_df.to_csv(summary_path, index=False)
olaf_details_df.to_csv(details_path, index=False)

print('Saved:', summary_path)
print('Saved:', details_path)

display(olaf_summary_df.T.rename(columns={0: 'value'}))


## Slide-ready table row

In [ ]:
slide_cols = ['series', 'STR', 'CR', 'PC', 'OC', 'CV', 'DVS']
slide_df = olaf_summary_df[slide_cols].copy()
for c in ['STR', 'CR', 'PC', 'OC', 'CV', 'DVS']:
    slide_df[c] = slide_df[c].astype(float)

display(slide_df)

latex_row = (
    f"OLAF & {slide_df.iloc[0]['STR']:.4f} & {slide_df.iloc[0]['CR']:.4f} & "
    f"{slide_df.iloc[0]['PC']:.4f} & {slide_df.iloc[0]['OC']:.4f} & "
    f"{slide_df.iloc[0]['CV']:.4f} & {slide_df.iloc[0]['DVS']:.4f} \\" 
)
print(latex_row)

latex_path = OUTPUT_DIR / 'olaf_semantic_validation_latex_row.txt'
latex_path.write_text(latex_row, encoding='utf-8')
print('Saved:', latex_path)


## Diagnostics: why each metric got its value

In [ ]:
print('Detected provenance columns:', olaf_summary.get('provenance_columns_detected'))
print('Detected support columns:', olaf_summary.get('support_columns_detected'))
print('Raw rows:', olaf_summary['raw_triple_row_count'])
print('Valid triples:', olaf_summary['valid_triple_count'])
print('Invalid/unresolved triples:', olaf_summary['invalid_or_unresolved_triple_count'])
print('Ontology-compliant triples:', olaf_summary['ontology_compliant_triple_count'])
print('Constraint violations among valid triples:', olaf_summary['constraint_violation_count'])

invalid_df = olaf_details_df[~olaf_details_df['valid_triple']].copy()
print('Invalid/unresolved examples:')
display(invalid_df.head(30))

misaligned_df = olaf_details_df[
    (olaf_details_df['valid_triple']) & (~olaf_details_df['ontology_compliant'])
].copy()
print('Ontology misalignment examples:')
display(misaligned_df.head(30))


## Optional: compare several systems in the same slide table

Add rows to `SYSTEMS` if you have other systems. Each item needs:

```python
{
    'series': 'NeoOLAF',
    'triples_path': Path(...),
    'ontology_path': Path(...),
    'source_text_path': Path(...) or None,
}
```

For NeoOLAF, this works best if the triple export contains provenance/support columns. If it only contains a compact KG JSON without provenance, PC/STR will be low by construction.


In [ ]:
SYSTEMS = [
    {
        'series': 'OLAF',
        'triples_path': OLAF_TRIPLES_PATH,
        'ontology_path': OLAF_ONTOLOGY_PATH,
        'source_text_path': OLAF_SOURCE_TEXT_PATH,
    },
    # Example:
    # {
    #     'series': 'NeoOLAF',
    #     'triples_path': RUNS_ROOT / 'xquality_machine32' / '...' / 'exports' / 'kg_inferred.json',
    #     'ontology_path': RUNS_ROOT / 'xquality_machine32' / '...' / 'exports' / 'ontology.ttl',
    #     'source_text_path': OLAF_SOURCE_TEXT_PATH,
    # },
]

all_rows = []
all_details = []
for spec in SYSTEMS:
    tpath = Path(spec['triples_path'])
    opath = Path(spec['ontology_path'])
    if not tpath.exists() or not opath.exists():
        print('Skipping missing system:', spec['series'], tpath, opath)
        continue
    df = load_triple_table(tpath)
    summary, details = compute_semantic_validation_metrics(
        df,
        ontology_path=opath,
        source_text_path=spec.get('source_text_path'),
        config=CONFIG,
    )
    summary['series'] = spec['series']
    summary['triple_path'] = str(tpath)
    summary['ontology_path'] = str(opath)
    all_rows.append(summary)
    details.insert(0, 'series', spec['series'])
    all_details.append(details)

multi_summary_df = pd.DataFrame(all_rows)
multi_details_df = pd.concat(all_details, ignore_index=True) if all_details else pd.DataFrame()

multi_summary_path = OUTPUT_DIR / 'semantic_validation_multi_system_summary.csv'
multi_details_path = OUTPUT_DIR / 'semantic_validation_multi_system_details.csv'
multi_summary_df.to_csv(multi_summary_path, index=False)
multi_details_df.to_csv(multi_details_path, index=False)

print('Saved:', multi_summary_path)
print('Saved:', multi_details_path)
display(multi_summary_df[[c for c in ['series','STR','CR','PC','OC','CV','DVS','raw_triple_row_count','valid_triple_count'] if c in multi_summary_df.columns]])


## Plot slide metrics

In [ ]:
import matplotlib.pyplot as plt

plot_df = multi_summary_df[['series', 'STR', 'CR', 'PC', 'OC', 'CV', 'DVS']].melt(
    id_vars='series', var_name='metric', value_name='score'
)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
for series, sub in plot_df.groupby('series'):
    ax.plot(sub['metric'], sub['score'], marker='o', label=series)
ax.set_ylim(-0.03, 1.03)
ax.set_title('Semantic validation metrics')
ax.set_ylabel('Score')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plot_path = OUTPUT_DIR / 'semantic_validation_metrics_plot.png'
fig.savefig(plot_path, dpi=180)
plt.show()
print('Saved:', plot_path)


## Suggested interpretation wording

Use wording like this:

> STR and PC evaluate whether the generated KG keeps explicit traceability/provenance to the source. OC and CV evaluate whether generated triples are structurally aligned with the generated ontology. CR penalizes unresolved or incomplete triples. DVS is a strict global validation indicator and should be interpreted as a pass/fail semantic validation score, not as extraction F1.

For OLAF specifically, if STR/PC are low, it means the generated triples may be readable but do not carry enough explicit provenance/support metadata for traceable semantic validation.
